# Indexing and Filtering DataFrames

Selecting and filtering data is what you'll do most often in Pandas. The `.loc` and `.iloc` indexers have different semantics that confuse many beginners. Boolean filtering with multiple conditions has gotchas you need to know.

## Learning Objectives

- Understand the difference between `.loc` (label-based) and `.iloc` (integer-based)
- Filter DataFrames with boolean conditions
- Combine multiple conditions correctly with `&`, `|`, `~`
- Use `.query()` for readable filtering
- Work with index: set, reset, and filter by index

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Load data
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

## .loc vs .iloc

| | `.loc` | `.iloc` |
|---|---|---|
| **Based on** | Labels (index values) | Integer positions |
| **Slicing** | Inclusive on both ends | Exclusive on end |
| **Syntax** | `df.loc[row_labels, col_labels]` | `df.iloc[row_positions, col_positions]` |

In [ ]:
# .iloc - integer position based
# Row 0, all columns
print("df.iloc[0] - First row:")
print(df.iloc[0])

In [ ]:
# .iloc - specific cell
print(f"df.iloc[0, 1] = {df.iloc[0, 1]}")  # Row 0, column 1

In [ ]:
# .iloc - slicing (end is EXCLUSIVE like Python)
print("df.iloc[0:3, 0:4] - Rows 0-2, Cols 0-3:")
df.iloc[0:3, 0:4]

In [ ]:
# .loc - label based
# With default integer index, labels ARE the integers
print("df.loc[0, 'Name'] - Row label 0, column 'Name':")
print(df.loc[0, 'Name'])

In [ ]:
# .loc - slicing (end is INCLUSIVE!)
print("df.loc[0:2, 'Name':'Age'] - Row labels 0-2, cols 'Name' to 'Age':")
df.loc[0:2, 'Name':'Age']

In [ ]:
# .loc with list of columns
print("df.loc[:5, ['Name', 'Age', 'Fare']]:")
df.loc[:5, ['Name', 'Age', 'Fare']]

> **Gotcha: .loc slice is inclusive!**  
> `df.loc[0:5]` includes row 5, but `df.iloc[0:5]` does NOT include row 5.  
> This trips up people coming from NumPy where slicing is always exclusive on the end.

In [ ]:
# Demonstrate the difference
print(f"df.iloc[0:3] returns {len(df.iloc[0:3])} rows (0, 1, 2)")
print(f"df.loc[0:3] returns {len(df.loc[0:3])} rows (0, 1, 2, 3)")

## Boolean Filtering

The most powerful way to filter data. Create a boolean mask and use it to select rows.

In [ ]:
# Single condition
# Find all passengers over 50
mask = df['Age'] > 50
print(f"Mask type: {type(mask)}")
print(f"Mask values: {mask.head(10).tolist()}")

# Apply mask
over_50 = df[mask]
print(f"\nPassengers over 50: {len(over_50)}")
over_50.head()

In [ ]:
# More concise: condition directly in brackets
over_50 = df[df['Age'] > 50]
print(f"Same result: {len(over_50)} passengers")

In [ ]:
# Multiple conditions with & (AND), | (OR), ~ (NOT)
# CRITICAL: Each condition MUST be in parentheses!

# Women in first class
women_first = df[(df['Sex'] == 'female') & (df['Pclass'] == 1)]
print(f"Women in 1st class: {len(women_first)}")

# Children (under 18) OR elderly (over 60)
children_or_elderly = df[(df['Age'] < 18) | (df['Age'] > 60)]
print(f"Children or elderly: {len(children_or_elderly)}")

# NOT survivors (died)
died = df[~(df['Survived'] == 1)]
print(f"Did not survive: {len(died)}")

> **Gotcha: Parentheses are required!**  
> `df[df['Age'] > 18 & df['Sex'] == 'male']` will ERROR!  
> Use: `df[(df['Age'] > 18) & (df['Sex'] == 'male')]`  
> This is because `&` has higher precedence than comparison operators.

In [ ]:
# Complex filtering
# Male survivors over 30 who paid more than $50
filtered = df[
    (df['Sex'] == 'male') & 
    (df['Survived'] == 1) & 
    (df['Age'] > 30) & 
    (df['Fare'] > 50)
]
print(f"Matching passengers: {len(filtered)}")
filtered[['Name', 'Age', 'Fare', 'Pclass']]

## The .query() Method

For complex filters, `.query()` can be more readable.

In [ ]:
# Same filter using .query()
filtered = df.query('Sex == "male" and Survived == 1 and Age > 30 and Fare > 50')
print(f"Using query: {len(filtered)} passengers")
filtered[['Name', 'Age', 'Fare', 'Pclass']]

In [ ]:
# .query() advantages:
# - No parentheses needed
# - No df[] prefix for each column
# - Uses natural 'and', 'or', 'not' keywords

# Using variables with @
min_age = 30
min_fare = 50
filtered = df.query('Age > @min_age and Fare > @min_fare')
print(f"Using variables: {len(filtered)} passengers")

## isin() for Membership Testing

In [ ]:
# Filter passengers from specific ports
ports = ['C', 'Q']
from_c_or_q = df[df['Embarked'].isin(ports)]
print(f"From Cherbourg or Queenstown: {len(from_c_or_q)}")
from_c_or_q['Embarked'].value_counts()

In [ ]:
# NOT in list: use ~
not_from_s = df[~df['Embarked'].isin(['S'])]
print(f"NOT from Southampton: {len(not_from_s)}")

## Working with Index

In [ ]:
# Set a column as index
df_indexed = df.set_index('PassengerId')
print("With PassengerId as index:")
df_indexed.head()

In [ ]:
# Now .loc uses PassengerId as the label
print("df_indexed.loc[1] - Passenger 1:")
df_indexed.loc[1]

In [ ]:
# Select multiple by index
df_indexed.loc[[1, 3, 5, 7, 9]]

In [ ]:
# Reset index back to default
df_reset = df_indexed.reset_index()
print("After reset_index():")
df_reset.head()

## Visualization: Comparing Filtered Groups

In [ ]:
# Compare age distributions: survivors vs non-survivors
fig, ax = plt.subplots(figsize=(10, 6))

survivors = df[df['Survived'] == 1]['Age'].dropna()
non_survivors = df[df['Survived'] == 0]['Age'].dropna()

bins = np.arange(0, 85, 5)

ax.hist(survivors, bins=bins, alpha=0.6, label=f'Survived (n={len(survivors)})', color='green')
ax.hist(non_survivors, bins=bins, alpha=0.6, label=f'Did Not Survive (n={len(non_survivors)})', color='red')

ax.set_xlabel('Age')
ax.set_ylabel('Count')
ax.set_title('Age Distribution: Survivors vs Non-Survivors')
ax.legend()

# Add mean lines
ax.axvline(survivors.mean(), color='green', linestyle='--', linewidth=2)
ax.axvline(non_survivors.mean(), color='red', linestyle='--', linewidth=2)

plt.tight_layout()
plt.show()

print(f"Mean age of survivors: {survivors.mean():.1f}")
print(f"Mean age of non-survivors: {non_survivors.mean():.1f}")

## Exercises

### Exercise 1: .loc vs .iloc Practice

Using the Titanic dataset:
1. Get the name of the passenger in row 100 using `.iloc`
2. Get rows 10-15 (inclusive) with columns Name, Age, Fare using `.loc`
3. Get the last 5 rows using `.iloc`

In [ ]:
# Practice indexing
# YOUR CODE HERE

### Exercise 2: Boolean Filtering

Find all passengers who:
1. Are female AND survived
2. Paid more than 100 for their fare OR are in first class
3. Are children (under 18) who did NOT survive

In [ ]:
# Boolean filtering practice
# YOUR CODE HERE

### Exercise 3: Using .query()

Rewrite these filters using `.query()`:
1. `df[(df['Age'] >= 20) & (df['Age'] <= 40)]`
2. `df[(df['Pclass'] == 1) | (df['Pclass'] == 2)]`

In [ ]:
# Convert to .query() syntax
# YOUR CODE HERE

### Exercise 4: Visualization Challenge

Create a histogram comparing the fare distributions of:
- Passengers who embarked at Southampton (S)
- Passengers who embarked elsewhere

Add a legend and appropriate labels.

In [ ]:
# Create the visualization
# YOUR CODE HERE